# 真实 aligned rescoring 运行入口

该 Notebook 用于 Colab 冷启动: 挂载 Google Drive、拉取仓库、加载 SD3.5 Medium、读取 ready attention geometry 包、重建 attention-relative carrier, 在真实 latent callback 中获取对齐前后 latent 投影, 重新计算 LF/HF 内容分数, 计算 LPIPS 与 CLIP pair-level 质量指标, 并把核对文件打包到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/aligned_rescoring.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 SD3.5 Medium 访问权限, 并配置 `HF_TOKEN`。
3. 确认 Drive 中已有 ready attention geometry 包, 默认查找 `/content/drive/MyDrive/SLM/attention_geometry/attention_geometry_package_*.zip`。
4. 默认输出目录为 `/content/drive/MyDrive/SLM/aligned_rescoring`。
5. 本入口要求 LPIPS 与 CLIP pair-level 指标可计算; FID / KID 仍保留为后续数据集级统计。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/aligned_rescoring')
os.environ.setdefault('SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR', '/content/drive/MyDrive/SLM/attention_geometry')
os.environ.setdefault('SLM_WM_ALIGNED_RESCORING_SUBSPACE_RECORDS', '16')
os.environ.setdefault('SLM_WM_ALIGNED_RESCORING_CARRIER_COUNT', '1')
os.environ.setdefault('SLM_WM_ATTENTION_RUNTIME_STRENGTH', '0.025')
os.environ.setdefault('SLM_WM_ENABLE_PAIR_PERCEPTUAL_METRICS', '1')
os.environ.setdefault('SLM_WM_REQUIRE_PAIR_PERCEPTUAL_METRICS', '1')
os.environ.setdefault('SLM_WM_CLIP_MODEL_ID', 'openai/clip-vit-base-patch32')
os.environ.setdefault('SLM_WM_LPIPS_NETWORK', 'alex')
os.environ.setdefault('SLM_WM_PERCEPTUAL_METRIC_DEVICE', 'cuda')
print({
    'drive_output_dir': os.environ['SLM_WM_DRIVE_OUTPUT_DIR'],
    'attention_geometry_drive_dir': os.environ['SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR'],
    'aligned_rescoring_subspace_records': os.environ['SLM_WM_ALIGNED_RESCORING_SUBSPACE_RECORDS'],
    'aligned_rescoring_carrier_count': os.environ['SLM_WM_ALIGNED_RESCORING_CARRIER_COUNT'],
    'attention_runtime_strength': os.environ['SLM_WM_ATTENTION_RUNTIME_STRENGTH'],
    'enable_pair_perceptual_metrics': os.environ['SLM_WM_ENABLE_PAIR_PERCEPTUAL_METRICS'],
    'require_pair_perceptual_metrics': os.environ['SLM_WM_REQUIRE_PAIR_PERCEPTUAL_METRICS'],
    'clip_model_id': os.environ['SLM_WM_CLIP_MODEL_ID'],
    'lpips_network': os.environ['SLM_WM_LPIPS_NETWORK'],
    'perceptual_metric_device': os.environ['SLM_WM_PERCEPTUAL_METRIC_DEVICE'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub
%pip install -q --upgrade lpips


In [ ]:
import importlib
import importlib.metadata as importlib_metadata
import importlib.util

# 只做只读依赖诊断, 不清理 sys.modules, 避免在同一 Colab 内核中重载 numpy / torch。
importlib.invalidate_caches()

import diffusers
import transformers
from diffusers import StableDiffusion3Pipeline
from transformers import CLIPModel, CLIPProcessor, CLIPTextModelWithProjection
from transformers.generation import GenerationMixin

def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'

dependency_report = {
    'diffusers': diffusers.__version__,
    'transformers': transformers.__version__,
    'numpy': package_version_or_missing('numpy'),
    'tokenizers': package_version_or_missing('tokenizers'),
    'accelerate': package_version_or_missing('accelerate'),
    'huggingface_hub': package_version_or_missing('huggingface_hub'),
    'lpips': package_version_or_missing('lpips'),
    'torchvision': package_version_or_missing('torchvision'),
    'scipy_available': importlib.util.find_spec('scipy') is not None,
    'torchvision_available': importlib.util.find_spec('torchvision') is not None,
    'lpips_available': importlib.util.find_spec('lpips') is not None,
    'stable_diffusion_pipeline': StableDiffusion3Pipeline.__name__,
    'clip_model': CLIPModel.__name__,
    'clip_processor': CLIPProcessor.__name__,
    'clip_text_model_with_projection': CLIPTextModelWithProjection.__name__,
    'generation_mixin': GenerationMixin.__name__,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from pathlib import Path

geometry_dir = Path(os.environ['SLM_WM_ATTENTION_GEOMETRY_DRIVE_DIR'])
geometry_packages = sorted(geometry_dir.glob('attention_geometry_package_*.zip'))
assert geometry_packages, f'未找到 attention geometry 包: {geometry_dir}'
print({'geometry_package': str(geometry_packages[-1])})


In [ ]:
from paper_workflow.colab_utils.aligned_rescoring import run_default_aligned_rescoring_plan

aligned_rescoring_summary = run_default_aligned_rescoring_plan(root='.')
assert aligned_rescoring_summary['run_decision'] == 'pass', aligned_rescoring_summary
assert aligned_rescoring_summary['aligned_rescoring_record_count'] > 0, aligned_rescoring_summary
assert aligned_rescoring_summary['real_aligned_rescore_count'] > 0, aligned_rescoring_summary
assert aligned_rescoring_summary['image_quality_metrics_ready'] is True, aligned_rescoring_summary
assert aligned_rescoring_summary['perceptual_metrics_ready'] is True, aligned_rescoring_summary
aligned_rescoring_summary


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.aligned_rescoring import package_aligned_rescoring_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ.get('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/aligned_rescoring')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'aligned_rescoring_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_aligned_rescoring_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/aligned_rescoring').glob('*.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/aligned_rescoring').glob('*.csv')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8')[:4000])
